# 06 Forecasting
Forecast engagement with Prophet when available; fallback to Exponential Smoothing.

In [1]:
import warnings
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils" / "utils.py").exists():
    if (PROJECT_ROOT / "notebooks" / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT / "notebooks"
    elif (PROJECT_ROOT.parent / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils import ensure_project_dirs, load_raw_dataset, clean_dataset, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR
from utils.features import engineer_kpis, build_post_feature_sets, aggregate_business_features
from utils.evaluation import regression_metrics, rank_models
from utils.visualization import set_plot_style, save_figure

set_plot_style()
ensure_project_dirs()
RAW_DATA_PATH = Path(r"C:\univ\Data mining\Project\synthetic_generator\synthetic_social_media_posts.csv")
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = PROJECT_ROOT / "synthetic_generator" / "synthetic_social_media_posts.csv"
KPI_PATH = PROCESSED_DIR / "kpi_dataset.csv"

## Load KPI and Build Weekly/Monthly Time Series

In [2]:
if KPI_PATH.exists():
    df = pd.read_csv(KPI_PATH, parse_dates=["post_date"])
else:
    df = engineer_kpis(clean_dataset(load_raw_dataset(RAW_DATA_PATH)))
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,is_reel,posting_time_bin,caption_length_bin,hashtags_count_bin,emoji_count_bin,engagement_level,business_size_bin,high_engagement,high_view_rate,high_comment_rate
0,Ramallah Care Pharmacy,pharmacy,13666,2025-12-16,22,Tuesday,12,carousel,صحتك بتهمنا Swipe وشوفوا الباقي. موجودين في Ra...,109,...,0,night,medium,few,none,low,large,0,0,0
1,Nablus Bites,restaurant,12838,2025-10-18,20,Saturday,10,video,اليوم عنا أكل طيب فيديو جديد. احنا جاهزين احجز...,78,...,0,evening,medium,few,none,medium,medium,0,0,0
2,Al Amal Dental,clinic,4652,2025-01-17,21,Friday,1,carousel,صحتكم أولويتنا سوايب وشوفوا التفاصيل. شو رأيكم...,142,...,0,night,long,few,few,low,small,0,0,0
3,Hebron Study Hub,education_center,26289,2025-03-27,20,Thursday,3,reel,استنونا بدورة جديدة شوفوا الريل. زورونا بفرعنا...,90,...,1,evening,medium,few,few,high,large,1,1,1
4,Bethlehem Brew Bar,cafe,2210,2025-11-01,22,Saturday,11,image,أجواء ولا أروع صورة جديدة. أهلا بالناس الحلوة ...,72,...,0,night,medium,few,none,medium,small,0,0,0


In [3]:
import importlib.util

def build_ts(frame, freq):
    ts = (
        frame.set_index("post_date")["engagement"]
        .resample(freq)
        .mean()
        .dropna()
        .reset_index()
        .rename(columns={"post_date": "ds", "engagement": "y"})
    )
    ts["ds"] = pd.to_datetime(ts["ds"])
    return ts

weekly = build_ts(df, "W-MON")
monthly = build_ts(df, "MS")


## Experiment Grid and Metrics

In [4]:
has_prophet = importlib.util.find_spec("prophet") is not None
if has_prophet:
    from prophet import Prophet
else:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing

cp_vals, modes = [0.01,0.05,0.1,0.5], ["additive","multiplicative"]
rows, store = [], {}
for agg_name, ts in {"weekly": weekly, "monthly": monthly}.items():
    ts = ts.sort_values("ds").reset_index(drop=True)
    if len(ts) < 8:
        continue
    split = int(len(ts) * 0.8)
    train, test = ts.iloc[:split], ts.iloc[split:]
    if len(test) == 0:
        continue
    if has_prophet:
        for cp in cp_vals:
            for mode in modes:
                m = Prophet(changepoint_prior_scale=cp, seasonality_mode=mode, yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=False)
                m.fit(train)
                fut = m.make_future_dataframe(periods=len(test), freq="W-MON" if agg_name=="weekly" else "MS")
                pred = m.predict(fut)[["ds","yhat"]].tail(len(test))
                mets = regression_metrics(test["y"], pred["yhat"])
                rows.append({"model":"prophet","aggregation":agg_name,"changepoint_prior_scale":cp,"seasonality_mode":mode, **mets})
                store[("prophet",agg_name,cp,mode)] = (train,test,pred)
    else:
        model = ExponentialSmoothing(train["y"], trend="add", seasonal=None).fit(optimized=True)
        pred = pd.DataFrame({"ds": test["ds"].values, "yhat": model.forecast(len(test)).values})
        mets = regression_metrics(test["y"], pred["yhat"])
        for cp in cp_vals:
            for mode in modes:
                rows.append({"model":"exp_smoothing_fallback","aggregation":agg_name,"changepoint_prior_scale":cp,"seasonality_mode":mode, **mets})
                store[("exp_smoothing_fallback",agg_name,cp,mode)] = (train,test,pred)

exp_input = pd.DataFrame(rows)
if exp_input.empty:
    exp = pd.DataFrame(columns=["model","aggregation","changepoint_prior_scale","seasonality_mode","MAE","RMSE","MAPE","composite_score"])
    best = None
    forecast = pd.DataFrame(columns=["ds","y","yhat","level","entity"])
else:
    exp = rank_models(exp_input, lower_is_better_cols=["MAE","RMSE","MAPE"])
    best = exp.iloc[0]
    train, test, pred = store[(best["model"],best["aggregation"],best["changepoint_prior_scale"],best["seasonality_mode"])]
    forecast = test.merge(pred, on="ds", how="left")
    forecast["level"], forecast["entity"] = "overall", "all_businesses"


## Sector/Business Extensions and Save

In [5]:
extra = []
if best is not None:
    freq = "W-MON" if best["aggregation"] == "weekly" else "MS"

    def build_segment_ts(seg, freq):
        ts = (
            seg.set_index("post_date")["engagement"]
            .resample(freq)
            .mean()
            .dropna()
            .reset_index()
            .rename(columns={"post_date": "ds", "engagement": "y"})
        )
        ts["ds"] = pd.to_datetime(ts["ds"])
        return ts

    for sector, seg in df.groupby("sector"):
        ts = build_segment_ts(seg, freq)
        if len(ts) < 10:
            continue
        split = int(len(ts)*0.8)
        train_s, test_s = ts.iloc[:split], ts.iloc[split:]
        if len(test_s) < 2:
            continue
        if best["model"] == "prophet":
            from prophet import Prophet
            m = Prophet(changepoint_prior_scale=float(best["changepoint_prior_scale"]), seasonality_mode=best["seasonality_mode"], yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=False)
            m.fit(train_s)
            fut = m.make_future_dataframe(periods=len(test_s), freq=freq)
            pred_s = m.predict(fut)[["ds","yhat"]].tail(len(test_s))
        else:
            from statsmodels.tsa.holtwinters import ExponentialSmoothing
            fit = ExponentialSmoothing(train_s["y"], trend="add", seasonal=None).fit(optimized=True)
            pred_s = pd.DataFrame({"ds": test_s["ds"].values, "yhat": fit.forecast(len(test_s)).values})
        mrg = test_s.merge(pred_s, on="ds", how="left")
        mrg["level"], mrg["entity"] = "sector", sector
        extra.append(mrg)

    for business, seg in df.groupby("business_name"):
        ts = build_segment_ts(seg, freq)
        if len(ts) < 12:
            continue
        split = int(len(ts)*0.8)
        train_b, test_b = ts.iloc[:split], ts.iloc[split:]
        if len(test_b) < 2:
            continue
        if best["model"] == "prophet":
            from prophet import Prophet
            m = Prophet(changepoint_prior_scale=float(best["changepoint_prior_scale"]), seasonality_mode=best["seasonality_mode"], yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=False)
            m.fit(train_b)
            fut = m.make_future_dataframe(periods=len(test_b), freq=freq)
            pred_b = m.predict(fut)[["ds","yhat"]].tail(len(test_b))
        else:
            from statsmodels.tsa.holtwinters import ExponentialSmoothing
            fit = ExponentialSmoothing(train_b["y"], trend="add", seasonal=None).fit(optimized=True)
            pred_b = pd.DataFrame({"ds": test_b["ds"].values, "yhat": fit.forecast(len(test_b)).values})
        mrg = test_b.merge(pred_b, on="ds", how="left")
        mrg["level"], mrg["entity"] = "business", business
        extra.append(mrg)

forecast = pd.concat([forecast] + extra, ignore_index=True) if extra else forecast
forecast.to_csv(PROCESSED_DIR / "forecast.csv", index=False)
exp.to_csv(PROCESSED_DIR / "forecast_metrics.csv", index=False)
display(exp.head(10))
display(forecast.head(20))
if best is None:
    print("No valid forecasting experiments were generated from available data.")
else:
    print("Insight: selected model balances low error and stable behavior across aggregation levels.")


C:\univ\Data mining\Project\notebooks\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
C:\univ\Data mining\Project\notebooks\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


C:\univ\Data mining\Project\notebooks\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


,model,aggregation,changepoint_prior_scale,seasonality_mode,MAE,RMSE,MAPE,composite_score
0,exp_smoothing_fallback,monthly,0.50,multiplicative,620.869183,687.168089,27.798291,3.0
1,exp_smoothing_fallback,monthly,0.50,additive,620.869183,687.168089,27.798291,3.0
2,exp_smoothing_fallback,monthly,0.10,multiplicative,620.869183,687.168089,27.798291,3.0
3,exp_smoothing_fallback,monthly,0.10,additive,620.869183,687.168089,27.798291,3.0
4,exp_smoothing_fallback,monthly,0.05,multiplicative,620.869183,687.168089,27.798291,3.0
5,exp_smoothing_fallback,monthly,0.05,additive,620.869183,687.168089,27.798291,3.0
6,exp_smoothing_fallback,monthly,0.01,multiplicative,620.869183,687.168089,27.798291,3.0
7,exp_smoothing_fallback,monthly,0.01,additive,620.869183,687.168089,27.798291,3.0
8,exp_smoothing_fallback,weekly,0.50,multiplicative,646.342860,815.284870,37.235285,0.0
9,exp_smoothing_fallback,weekly,0.50,additive,646.342860,815.284870,37.235285,0.0


,ds,y,yhat,level,entity
0,2025-10-01,2117.275862,1558.115666,overall,all_businesses
1,2025-11-01,2514.111111,1505.699671,overall,all_businesses
2,2025-12-01,1748.319588,1453.283676,overall,all_businesses
3,2025-10-01,582.500000,374.827426,sector,bakery
4,2025-11-01,1197.714286,359.882272,sector,bakery
5,2025-12-01,1008.555556,344.937118,sector,bakery
6,2025-10-01,601.166667,483.073918,sector,beauty_salon
7,2025-11-01,437.500000,392.865964,sector,beauty_salon
8,2025-12-01,452.777778,302.658010,sector,beauty_salon
9,2025-10-01,590.750000,217.216179,sector,cafe


Insight: selected model balances low error and stable behavior across aggregation levels.
